In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PREPROCESSED_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap\training_set_rel3_preprocessed.csv"
)
PROMPT6_TRAITS_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-6.csv"
)

OUTPUT_PATH = Path(
    r"C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-6-concat.csv"
)


In [3]:
def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    raise ValueError(f"Unsupported file type: {path.suffix}")

In [4]:
pre = read_table(PREPROCESSED_PATH)
p6_traits = read_table(PROMPT6_TRAITS_PATH)

print("Preprocessed columns:")
print(pre.columns.tolist())

print("\nPrompt 6 trait columns:")
print(p6_traits.columns.tolist())

Preprocessed columns:
['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']

Prompt 6 trait columns:
['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']


In [5]:
pre = pre.rename(columns={
    "Essay ID": "essay_id",
    "essay_id": "essay_id",
    "Essay Set": "essay_set",
    "essay_set": "essay_set",
    "Essay": "essay",
    "essay": "essay",
})

p6_traits = p6_traits.rename(columns={
    "Essay ID": "essay_id",
    "Content": "p6_content",
    "Prompt Adherence": "p6_prompt_adherence",
    "Language": "p6_language",
    "Narrativity": "p6_narrativity",
})

In [6]:
required_pre_cols = [
    "essay_id",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm",
]

trait_cols = [
    "p6_content",
    "p6_prompt_adherence",
    "p6_language",
    "p6_narrativity",
]

required_trait_cols = ["essay_id"] + trait_cols

missing_pre = [c for c in required_pre_cols if c not in pre.columns]
missing_traits = [c for c in required_trait_cols if c not in p6_traits.columns]

if missing_pre:
    raise ValueError(f"Missing columns in preprocessed file: {missing_pre}")

if missing_traits:
    raise ValueError(
        f"Missing columns in prompt 6 trait file: {missing_traits}\n"
        f"Available columns after rename: {p6_traits.columns.tolist()}"
    )


In [7]:
pre_p6 = pre.loc[pre["essay_set"] == 6, required_pre_cols].copy()

pre_p6["essay_id"] = pre_p6["essay_id"].astype(int)
p6_traits["essay_id"] = p6_traits["essay_id"].astype(int)

p6_traits = p6_traits[required_trait_cols].copy()

print("\nRows in preprocessed prompt 6:", len(pre_p6))
print("Rows in prompt 6 trait file:", len(p6_traits))


Rows in preprocessed prompt 6: 1800
Rows in prompt 6 trait file: 1800


In [8]:
dup_traits = p6_traits[
    p6_traits["essay_id"].duplicated(keep=False)
].sort_values("essay_id")

print("\nSố dòng bị trùng essay_id trong p6_traits:", len(dup_traits))
print("Số essay_id bị trùng:", dup_traits["essay_id"].nunique())

if len(dup_traits) > 0:
    display(dup_traits.head(20))


Số dòng bị trùng essay_id trong p6_traits: 0
Số essay_id bị trùng: 0


In [9]:
p6_traits = p6_traits.drop_duplicates()

if p6_traits["essay_id"].duplicated().any():
    print("\nWarning: p6_traits has duplicated essay_id. Aggregating by mean.")
    p6_traits = (
        p6_traits
        .groupby("essay_id", as_index=False)[trait_cols]
        .mean()
    )

# Round half-up nếu có nhiều annotation/rater
for col in trait_cols:
    p6_traits[col] = np.floor(p6_traits[col] + 0.5).astype(int)

print("Duplicate essay_id còn lại:", p6_traits["essay_id"].duplicated().sum())

Duplicate essay_id còn lại: 0


In [10]:
benchmark_p6 = pre_p6.merge(
    p6_traits,
    on="essay_id",
    how="inner",
    validate="one_to_one"
)

benchmark_p6 = benchmark_p6.dropna(subset=trait_cols).copy()

benchmark_p6.insert(
    loc=2,
    column="prompt_id",
    value=6
)

In [11]:
benchmark_p6 = benchmark_p6[
    [
        "essay_id",
        "essay_set",
        "prompt_id",
        "essay",
        "domain1_score",
        "domain1_score_norm",
        "p6_content",
        "p6_prompt_adherence",
        "p6_language",
        "p6_narrativity",
    ]
]

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
benchmark_p6.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("\nDone.")
print(f"Prompt 6 rows in preprocessed: {len(pre_p6)}")
print(f"Prompt 6 rows after merge/dropna: {len(benchmark_p6)}")
print(f"Saved to: {OUTPUT_PATH}")

display(benchmark_p6.head())


Done.
Prompt 6 rows in preprocessed: 1800
Prompt 6 rows after merge/dropna: 1800
Saved to: C:\Users\PC\IELTS-Writing-Evals\dataset\asap++\Prompt-6-concat.csv


,essay_id,essay_set,prompt_id,essay,domain1_score,domain1_score_norm,p6_content,p6_prompt_adherence,p6_language,p6_narrativity
0,14834,6,6,There were many obstacles that the builders fa...,2.0,0.50,2,2,3,3
1,14835,6,6,"Him from the start, there would have been many...",3.0,0.75,4,3,2,2
2,14836,6,6,The builders of the Empire State Building face...,4.0,1.00,4,4,3,3
3,14837,6,6,In the passage The Mooring Mast by Marcia Amid...,1.0,0.25,1,1,2,1
4,14838,6,6,The builders of the Empire State Building face...,3.0,0.75,3,3,4,4
